In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
import torch
from PIL import Image
from open_clip import create_model_from_pretrained, get_tokenizer, create_model_and_transforms
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from transformers import AutoModel, AutoTokenizer
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from torch.utils.data import TensorDataset, DataLoader, Dataset
import torch.nn as nn
import torch.optim as optim
import plotly as px
from plotly import graph_objects as go
from sklearn.preprocessing import StandardScaler
from peft import LoraConfig, get_peft_model

In [2]:
pd.options.display.max_columns = None
sample_dataset = pd.read_csv("./chexpert_sample_dataset.csv", index_col=0)
sample_dataset

,path_to_image,deid_patient_id,report,sub_report,target,report_length,sub_report_length
0,train/patient18877/study1/view1_frontal.jpg,patient18877,NARRATIVE:\nCHEST: Two views.\nCOMPARISON: 12-...,\nThere is plate-like atelectasis seen at the ...,0,110,76
1,train/patient28169/study1/view1_frontal.jpg,patient28169,IMPRESSION:\n \nCARDIAC SILHOUETTE IS AT THE U...,\n \nCARDIAC SILHOUETTE IS AT THE UPPER LIMITS...,1,95,74
2,train/patient58769/study1/view1_frontal.jpg,patient58769,NARRATIVE:\nChest 1 View: 1-11-2020\n \nHISTOR...,\n \nUnchanged right internal jugular venous ...,1,86,48
3,train/patient54916/study1/view1_frontal.jpg,patient54916,NARRATIVE:\nChest 1 View: 6/6/2008\n \nHISTORY...,\n \n1. PORTABLE UPRIGHT FRONTAL CHEST RADIO...,1,100,68
4,train/patient01990/study1/view1_frontal.jpg,patient01990,NARRATIVE:\nCHEST:\nCLINICAL HISTORY: Check li...,\nThere is an endotracheal tube with its tip a...,1,152,118
...,...,...,...,...,...,...,...
21995,train/patient08833/study1/view1_frontal.jpg,patient08833,NARRATIVE:\nRADIOGRAPHIC EXAMINATION OF THE CH...,\n \nRedemonstration of right IJ central venou...,0,109,57
21996,train/patient63772/study1/view1_frontal.jpg,patient63772,"NARRATIVE:\nChest 1 View Portable, 09-01-14\n ...",\n \n1.STABLE POSITIONING OF THE LEFT CHEST ...,0,121,75
21997,train/patient46004/study1/view1_frontal.jpg,patient46004,"NARRATIVE:\nEXAM: Chest Post Needle Biopsy, 20...",\n \n1. SEMI-UPRIGHT FRONTAL VIEW OF THE CHEST...,1,101,59
21998,train/patient07484/study1/view1_frontal.jpg,patient07484,NARRATIVE:\nSINGLE VIEW OF THE CHEST: 9-28-20...,\n \n UPRIGHT PORTABLE CHEST WITHOUT COMPARIS...,1,64,35


In [3]:
X = sample_dataset.drop(columns=["target"])
y = sample_dataset["target"]

# Split with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

In [39]:
# Load the BiomedCLIP model and processor
model, preprocess = create_model_from_pretrained('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
tokenizer = get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)
model.eval()

CustomTextCLIP(
  (visual): TimmModel(
    (trunk): VisionTransformer(
      (patch_embed): PatchEmbed(
        (proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
        (norm): Identity()
      )
      (pos_drop): Dropout(p=0.0, inplace=False)
      (patch_drop): Identity()
      (norm_pre): Identity()
      (blocks): Sequential(
        (0): Block(
          (norm1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (attn): Attention(
            (qkv): Linear(in_features=768, out_features=2304, bias=True)
            (q_norm): Identity()
            (k_norm): Identity()
            (attn_drop): Dropout(p=0.0, inplace=False)
            (norm): Identity()
            (proj): Linear(in_features=768, out_features=768, bias=True)
            (proj_drop): Dropout(p=0.0, inplace=False)
          )
          (ls1): Identity()
          (drop_path1): Identity()
          (norm2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): Mlp(
          

# Cross Attention + MLP

In [20]:
def biomed_clip_cross_att(image_path, report_text):
    # Prepare inputs
    image = preprocess(Image.open(image_path)).unsqueeze(0).to(device)
    text = tokenizer([report_text]).to(device)
    
    with torch.no_grad():
            # 1. Extract Image Patches Embeddings
            # Output shape: [Batch, Num_Patches, Vision_Dim] (e.g., [B, 197, 768])
            # 197 includes 196 patches (14x14) + 1 CLS token
            img_patches = model.visual.trunk.forward_features(image)
            img_patches = img_patches[:, 1:, :] 

            # Extracts sequence embeddings from PubMedBERT before pooler squashing.
            # Input shape: [Batch_Size, Max_Sequence_Length] -> (usually 256 for BiomedCLIP)
            # Output shape: [Batch_Size, Max_Sequence_Length, Text_Dimension] -> e.g., [B, 256, 640]
            text_outputs = model.text.transformer(text)
            text_tokens = text_outputs.last_hidden_state

            img_patches_np = img_patches.squeeze(0).cpu().numpy()
            text_tokens_np = text_tokens.squeeze(0).cpu().numpy()
            
    return img_patches_np, text_tokens_np

In [21]:
import gc

cross_att_X_train_txt = []
cross_att_X_train_img = []
for i, (img, txt) in enumerate(zip(("./CheXpert/CheXpert-v1.0-small/" + X_train["path_to_image"]).tolist(), X_train["report"].tolist())):
    img_pat, txt_pat = biomed_clip_cross_att(img, txt)
    cross_att_X_train_img.append(img_pat)
    cross_att_X_train_txt.append(txt_pat)

    if i % 200 == 0 and i > 0:
        gc.collect()
        torch.cuda.empty_cache()

cross_att_y_train = y_train.to_numpy()


In [22]:
cross_att_X_train_txt = np.array(cross_att_X_train_txt)
cross_att_X_train_img = np.array(cross_att_X_train_img)

In [23]:
cross_att_X_test_txt = []
cross_att_X_test_img = []
for i, (img, txt) in enumerate(zip(("./CheXpert/CheXpert-v1.0-small/" + X_test["path_to_image"]).tolist(), X_test["report"].tolist())):
    img_pat, txt_pat = biomed_clip_cross_att(img, txt)
    cross_att_X_test_img.append(img_pat)
    cross_att_X_test_txt.append(txt_pat)

    if i % 200 == 0 and i > 0:
        gc.collect()
        torch.cuda.empty_cache()

cross_att_y_test = y_test.to_numpy()

In [24]:
cross_att_X_test_txt = np.array(cross_att_X_test_txt)
cross_att_X_test_img = np.array(cross_att_X_test_img)

In [25]:
cross_att_X_train_txt_tensor = torch.from_numpy(cross_att_X_train_txt).float()
cross_att_X_train_img_tensor = torch.from_numpy(cross_att_X_train_img).float()
cross_att_y_train_tensor = torch.from_numpy(cross_att_y_train).float()
cross_att_train_dataset = TensorDataset(cross_att_X_train_img_tensor, cross_att_X_train_txt_tensor, cross_att_y_train_tensor)
cross_att_X_test_txt_tensor = torch.from_numpy(cross_att_X_test_txt).float()
cross_att_X_test_img_tensor = torch.from_numpy(cross_att_X_test_img).float()
cross_att_y_test_tensor = torch.from_numpy(cross_att_y_test).float()
cross_att_test_dataset = TensorDataset(cross_att_X_test_img_tensor, cross_att_X_test_txt_tensor, cross_att_y_test_tensor)

cross_att_train_loader = DataLoader(cross_att_train_dataset, batch_size=64, shuffle=True)
cross_att_val_loader = DataLoader(cross_att_test_dataset, batch_size=64, shuffle=False)

In [4]:
class CrossAttentionClassifier(nn.Module):
    def __init__(self, embed_dim=768, num_heads=8, dropout=0.3):
        super().__init__()
        
        self.cross_attention = nn.MultiheadAttention(
                    embed_dim=embed_dim, 
                    num_heads=num_heads, 
                    dropout=dropout,
                    batch_first=True
                )
        
        self.layer_norm1 = nn.LayerNorm(embed_dim)
        self.layer_norm2 = nn.LayerNorm(embed_dim)
    
        # Add a small Feed-Forward Network layer (FFN) inside the attention block 
        # This mirrors a true standard Transformer block and stabilizes representation warping
        self.ffn = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim * 2, embed_dim)
        )
    
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.GELU(),
            nn.LayerNorm(256), # BatchNorm stabilizes linear scaling transitions!
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )
        
    def forward(self, img_patches, text_tokens):
        norm_text = self.layer_norm1(text_tokens)
        # Cross-Attention Core
        attn_output, _ = self.cross_attention(
            query=norm_text, 
            key=img_patches, 
            value=img_patches
        )
        x = (attn_output + text_tokens)
        
        # FFN stabilization step
        x = self.layer_norm2(self.ffn(x)) + x
        
        # Step 3: Max-pooling across sequence dimension 
        # This captures the strongest text-visual features and ignores inactive padding tokens
        x_pooled, _ = torch.max(x, dim=1)
        # Step 4: Predict
        return self.classifier(x_pooled)

In [5]:
def cross_attention_train(model, train_loader, val_loader, criterion, optimizer, scheduler, epochs=100):
    """
    Function to train a PyTorch model with training and validation datasets.

    Parameters:
    model: The neural network model to train.
    train_loader: DataLoader for the training dataset.
    val_loader: DataLoader for the validation dataset.
    criterion: Loss function (e.g., Binary Cross Entropy for classification).
    optimizer: Optimization algorithm (e.g., Adam, SGD).
    epochs: Number of training epochs (default=100).

    Returns:
    history: Dictionary containing loss and accuracy for both training and validation.
    """

    # Dictionary to store training & validation loss and accuracy over epochs
    history = {'loss': [], 'val_loss': [], 'accuracy': [], 'val_accuracy': []}

    for epoch in range(epochs):  # Loop over the number of epochs
        model.train()  # Set model to training mode
        total_loss, correct = 0, 0  # Initialize total loss and correct predictions

        # Training loop
        for inputs_img, inputs_txt, labels in train_loader:
            inputs_img = inputs_img.to(device)
            inputs_txt = inputs_txt.to(device)
            labels = labels.to(device)
            optimizer.zero_grad()  # Reset gradients before each batch
            outputs = model(inputs_img, inputs_txt).squeeze(1)  # Forward pass
            loss = criterion(outputs, labels)  # Compute loss
            loss.backward()  # Backpropagation (compute gradients)
            optimizer.step()  # Update model parameters

            total_loss += loss.item()  # Accumulate batch loss
            correct += ((torch.sigmoid(outputs) >= 0.5).float() == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for training
        train_loss = total_loss / len(train_loader)
        train_acc = correct / len(train_loader.dataset)

        # Validation phase (without gradient computation)
        model.eval()  # Set model to evaluation mode
        val_loss, val_correct = 0, 0
        with torch.no_grad():  # No need to compute gradients during validation
            for inputs_img, inputs_txt, labels in val_loader:
                inputs_img = inputs_img.to(device)
                inputs_txt = inputs_txt.to(device)
                labels = labels.to(device)
                outputs = model(inputs_img, inputs_txt).squeeze(1) # Forward pass
                loss = criterion(outputs, labels)  # Compute loss
                val_loss += loss.item()  # Accumulate validation loss
                val_correct += ((torch.sigmoid(outputs) >= 0.5).float() == labels).sum().item()  # Count correct predictions

        # Compute average loss and accuracy for validation
        val_loss /= len(val_loader)
        val_acc = val_correct / len(val_loader.dataset)
        
        scheduler.step()
        # Capture current learning rate for tracking
        current_lr = optimizer.param_groups[0]['lr']

        # Store metrics in history dictionary
        history['loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['accuracy'].append(train_acc)
        history['val_accuracy'].append(val_acc)

        # Print training progress
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {train_loss:.4f}, Acc: {train_acc:.4f}, "
              f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}, LR: {current_lr:.6f}")

    return history  # Return training history


In [28]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

cross_att_model = CrossAttentionClassifier().to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(cross_att_model.parameters(), lr=1e-5, weight_decay=1e-2)
# Scheduler forces learning rate to smoothly decay, ending the bouncing oscillations
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

# 5. Clear cache just to be safe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# 6. Launch the training block
history = cross_attention_train(
    model=cross_att_model, 
    train_loader=cross_att_train_loader, 
    val_loader=cross_att_val_loader, 
    criterion=criterion, 
    optimizer=optimizer, 
    scheduler = scheduler,
    epochs=50
)

Epoch [1/50], Loss: 0.6727, Acc: 0.5763, Val Loss: 0.5783, Val Acc: 0.7105, LR: 0.000010
Epoch [2/50], Loss: 0.5825, Acc: 0.6973, Val Loss: 0.5351, Val Acc: 0.7373, LR: 0.000010
Epoch [3/50], Loss: 0.5422, Acc: 0.7270, Val Loss: 0.5089, Val Acc: 0.7573, LR: 0.000010
Epoch [4/50], Loss: 0.5105, Acc: 0.7514, Val Loss: 0.4876, Val Acc: 0.7682, LR: 0.000010
Epoch [5/50], Loss: 0.4856, Acc: 0.7698, Val Loss: 0.4775, Val Acc: 0.7698, LR: 0.000010
Epoch [6/50], Loss: 0.4706, Acc: 0.7746, Val Loss: 0.4615, Val Acc: 0.7775, LR: 0.000010
Epoch [7/50], Loss: 0.4554, Acc: 0.7848, Val Loss: 0.4542, Val Acc: 0.7811, LR: 0.000010
Epoch [8/50], Loss: 0.4377, Acc: 0.7945, Val Loss: 0.4420, Val Acc: 0.7893, LR: 0.000009
Epoch [9/50], Loss: 0.4225, Acc: 0.8057, Val Loss: 0.4333, Val Acc: 0.7927, LR: 0.000009
Epoch [10/50], Loss: 0.4065, Acc: 0.8130, Val Loss: 0.4432, Val Acc: 0.7900, LR: 0.000009
Epoch [11/50], Loss: 0.3950, Acc: 0.8209, Val Loss: 0.4261, Val Acc: 0.7966, LR: 0.000009
Epoch [12/50], Loss

In [30]:
fig = go.Figure(data=[
    go.Scatter(y=history["loss"], name="Training Loss", mode="lines"),
    go.Scatter(y=history["val_loss"], name="Validation Loss", mode="lines")
])
fig.update_layout(title="Training and Validation Loss", xaxis_title="Epochs", yaxis_title="Loss")
fig.show()